### Cell 1 — Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

# Feature Selection
from sklearn.feature_selection import chi2, f_classif, mutual_info_classif
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Multicollinearity
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Class Imbalance
from imblearn.combine import SMOTETomek

# Settings
os.chdir('/Users/cirrus/Desktop/PMOS_PROJECT/notebook')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

print('✅ All libraries imported successfully')

✅ All libraries imported successfully


### Cell 2 — Load Data from Block 1

In [7]:
df = pd.read_csv('../data/processed/pmos_eda_clean.csv')

print(f'✅ Data loaded: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'\nTarget distribution:')
print(df['PCOS (Y/N)'].value_counts())

✅ Data loaded: 541 rows × 46 columns

Target distribution:
PCOS (Y/N)
0    364
1    177
Name: count, dtype: int64


### Cell 3 — Clean Column Names

In [12]:
# Strip all leading/trailing spaces from column names
df.columns = df.columns.str.strip()

print('✅ Column names cleaned')
print('\nAll columns after cleaning:')
for i, col in enumerate(df.columns, 1):
    print(f'{i:2d}. "{col}"')

✅ Column names cleaned

All columns after cleaning:
 1. "PCOS (Y/N)"
 2. "Age (yrs)"
 3. "Weight (Kg)"
 4. "Height(Cm)"
 5. "BMI"
 6. "Pulse rate(bpm)"
 7. "RR (breaths/min)"
 8. "Hb(g/dl)"
 9. "Cycle(R/I)"
10. "Cycle length(days)"
11. "Marraige Status (Yrs)"
12. "No. of aborptions"
13. "I   beta-HCG(mIU/mL)"
14. "II    beta-HCG(mIU/mL)"
15. "FSH(mIU/mL)"
16. "LH(mIU/mL)"
17. "FSH/LH"
18. "Hip(inch)"
19. "Waist(inch)"
20. "Waist:Hip Ratio"
21. "TSH (mIU/L)"
22. "AMH(ng/mL)"
23. "PRL(ng/mL)"
24. "Vit D3 (ng/mL)"
25. "PRG(ng/mL)"
26. "RBS(mg/dl)"
27. "Weight gain(Y/N)"
28. "hair growth(Y/N)"
29. "Skin darkening (Y/N)"
30. "Hair loss(Y/N)"
31. "Pimples(Y/N)"
32. "Fast food (Y/N)"
33. "Reg.Exercise(Y/N)"
34. "BP _Systolic (mmHg)"
35. "BP _Diastolic (mmHg)"
36. "Follicle No. (L)"
37. "Follicle No. (R)"
38. "Avg. F size (L) (mm)"
39. "Avg. F size (R) (mm)"
40. "Endometrium (mm)"
41. "LH_FSH_Ratio"


### Cell 4 — Fix Dtype Issues

In [13]:
# Check object columns
object_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Object columns found: {object_cols}')

# Convert to numeric
for col in object_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f'  ✅ {col} → converted to float | Missing created: {df[col].isnull().sum()}')

print(f'\nObject columns remaining: {df.select_dtypes(include="object").columns.tolist()}')

Object columns found: ['II    beta-HCG(mIU/mL)']
  ✅ II    beta-HCG(mIU/mL) → converted to float | Missing created: 1

Object columns remaining: []


### Cell 5 — Impute Missing Values

In [14]:
missing = df.isnull().sum()
missing = missing[missing > 0]
print('=== MISSING VALUES BEFORE IMPUTATION ===')
print(missing)

for col in missing.index:
    if df[col].nunique() <= 2:
        fill_val = df[col].mode()[0]
        strategy = 'mode'
    else:
        fill_val = df[col].median()
        strategy = 'median'
    
    df[col] = df[col].fillna(fill_val)
    print(f'  ✅ {col} → imputed with {strategy} ({fill_val:.3f})')

print(f'\nTotal missing values remaining: {df.isnull().sum().sum()}')

=== MISSING VALUES BEFORE IMPUTATION ===
Marraige Status (Yrs)     1
II    beta-HCG(mIU/mL)    1
AMH(ng/mL)                1
Fast food (Y/N)           1
dtype: int64
  ✅ Marraige Status (Yrs) → imputed with median (7.000)
  ✅ II    beta-HCG(mIU/mL) → imputed with median (1.990)
  ✅ AMH(ng/mL) → imputed with median (3.700)
  ✅ Fast food (Y/N) → imputed with mode (1.000)

Total missing values remaining: 0


### Cell 6 — Cap Extreme Outliers

In [15]:
clinical_caps = {
    'FSH(mIU/mL)'        : (0, 200),
    'LH(mIU/mL)'         : (0, 200),
    'FSH/LH'             : (0, 20),
    'LH_FSH_Ratio'       : (0, 20),
    'Pulse rate(bpm)'    : (40, 120),
    'TSH (mIU/L)'        : (0, 30),
    'AMH(ng/mL)'         : (0, 50),
    'Vit D3 (ng/mL)'     : (0, 150),
    'RBS(mg/dl)'         : (50, 300),
    'PRG(ng/mL)'         : (0, 30),
}

print('=== CAPPING EXTREME OUTLIERS ===')
for col, (low, high) in clinical_caps.items():
    if col in df.columns:
        before_max = df[col].max()
        before_min = df[col].min()
        df[col] = df[col].clip(lower=low, upper=high)
        print(f'  {col}: [{before_min:.2f}, {before_max:.2f}] → [{df[col].min():.2f}, {df[col].max():.2f}]')

print('\n✅ Capping complete')

=== CAPPING EXTREME OUTLIERS ===
  FSH(mIU/mL): [0.21, 5052.00] → [0.21, 200.00]
  LH(mIU/mL): [0.02, 2018.00] → [0.02, 200.00]
  FSH/LH: [0.00, 1372.83] → [0.00, 20.00]
  LH_FSH_Ratio: [0.00, 466.05] → [0.00, 20.00]
  Pulse rate(bpm): [13.00, 82.00] → [40.00, 82.00]
  TSH (mIU/L): [0.04, 65.00] → [0.04, 30.00]
  AMH(ng/mL): [0.10, 66.00] → [0.10, 50.00]
  Vit D3 (ng/mL): [0.00, 6014.66] → [0.00, 150.00]
  RBS(mg/dl): [60.00, 350.00] → [60.00, 300.00]
  PRG(ng/mL): [0.05, 85.00] → [0.05, 30.00]

✅ Capping complete
